# La récursivité
*NSI — Programmation*

**Objectifs :**
- comprendre ce qu'est une **fonction récursive** : une fonction qui s'appelle elle-même ;
- savoir repérer le **cas de base** et le **cas récursif** ;
- **suivre** l'exécution d'une fonction récursive (la trace, la pile d'appels) ;
- savoir **écrire** une fonction récursive avec une méthode en 3 questions ;
- appliquer la récursivité aux nombres, aux listes et aux chaînes de caractères.

**Prérequis :** les fonctions (paramètres, `return`), les conditions `if`, les listes et les chaînes de caractères.

Exécuter les cellules **dans l'ordre** (*Cell → Run All*), et ne pas hésiter à modifier les exemples.

---
# 1. L'idée

Tu fais la queue au cinéma et tu veux savoir **combien de personnes sont devant toi**, mais tu ne vois pas le début de la file. Tu demandes à la personne devant toi : « Combien de personnes sont devant toi ? ». Elle pose la même question à la personne devant elle, et ainsi de suite...

- La **première** personne de la file n'a personne devant elle : elle répond directement **0**. C'est le **cas de base**.
- Chaque personne reçoit une réponse, **ajoute 1** (pour la personne qu'elle a devant elle), et transmet la réponse en arrière. C'est le **cas récursif**.

Chacun résout **le même problème**, mais **un peu plus petit** (la file devant lui est plus courte d'une personne), jusqu'au cas où la réponse est évidente.

En programmation, une fonction **récursive** est une fonction qui **s'appelle elle-même** sur un problème plus petit.

---
# 2. Un premier exemple : le compte à rebours

In [1]:
def compte_a_rebours(n):
    if n == 0:                       # cas de base : on s'arrête
        print("Partez !")
    else:                            # cas récursif
        print(n)
        compte_a_rebours(n - 1)      # la fonction s'appelle elle-même, avec n - 1


compte_a_rebours(5)

5
4
3
2
1
Partez !


#### 💡 À retenir
Une fonction récursive a **toujours** deux parties :
- le **cas de base** : une situation simple où on répond **sans** appel récursif (ici `n == 0`) ;
- le **cas récursif** : on appelle la fonction **sur un problème plus petit** (ici `n - 1`), qui se rapproche du cas de base.

**Et sans cas de base ?** La fonction s'appellerait à l'infini. Python s'arrête au bout d'environ 1000 appels avec l'erreur :
```
RecursionError: maximum recursion depth exceeded
```
(« profondeur maximale de récursion dépassée »). Même erreur si le problème ne se rapproche jamais du cas de base, par exemple avec `compte_a_rebours(n + 1)` ou en appelant `compte_a_rebours(-1)`.

---
# 3. Une fonction récursive qui renvoie une valeur

On veut calculer la somme $1 + 2 + \cdots + n$. On remarque que :

$$\text{somme}(n) = n + \underbrace{(1 + 2 + \cdots + (n-1))}_{\text{somme}(n-1)} \qquad \text{et} \qquad \text{somme}(0) = 0$$

In [2]:
def somme(n):
    if n == 0:
        return 0                     # cas de base
    return n + somme(n - 1)          # cas récursif


print(somme(4))
print(somme(100))

10
5050


## 3.1 Suivre l'exécution : la trace

Pour bien comprendre, on ajoute des `print` qui montrent chaque appel. Le paramètre `decalage` est une chaîne d'espaces qui s'allonge à chaque appel, pour décaler l'affichage vers la droite (il ne sert qu'à l'affichage) :

In [3]:
def somme_trace(n, decalage):
    print(decalage + "appel de somme(" + str(n) + ")")
    if n == 0:
        print(decalage + "cas de base : somme(0) renvoie 0")
        return 0
    resultat = n + somme_trace(n - 1, decalage + "    ")
    print(decalage + "somme(" + str(n) + ") renvoie " + str(n) + " + somme(" + str(n - 1) + ") = " + str(resultat))
    return resultat


somme_trace(3, "")

appel de somme(3)
    appel de somme(2)
        appel de somme(1)
            appel de somme(0)
            cas de base : somme(0) renvoie 0
        somme(1) renvoie 1 + somme(0) = 1
    somme(2) renvoie 2 + somme(1) = 3
somme(3) renvoie 3 + somme(2) = 6


6

#### 💡 À retenir : la descente, puis la remontée
- **La descente :** `somme(3)` a besoin de `somme(2)`, qui a besoin de `somme(1)`, qui a besoin de `somme(0)`. Aucun calcul n'est encore terminé : chaque appel **attend** le résultat du suivant.
- **Le cas de base** répond directement : `0`.
- **La remontée :** chaque appel en attente reçoit sa réponse, termine **son** calcul et renvoie le résultat à celui qui l'a appelé : `1`, puis `3`, puis `6`.

Pour **voir** les appels s'empiler, coller le code de `somme` sur [pythontutor.com](https://pythontutor.com) et avancer pas à pas.

---
# 4. La pile d'appels

Pour se souvenir de tous les appels **en attente**, Python les range dans une **pile** (comme une pile d'assiettes) :

```
   La descente : les appels s'empilent         La remontée : les appels se dépilent

   │ somme(0) │ ← sommet : renvoie 0
   │ somme(1) │   en attente                  │ somme(1) │ reçoit 0, renvoie 1 + 0 = 1
   │ somme(2) │   en attente                  │ somme(2) │ reçoit 1, renvoie 2 + 1 = 3
   │ somme(3) │   en attente                  │ somme(3) │ reçoit 3, renvoie 3 + 3 = 6
   └──────────┘                               └──────────┘
```

- Chaque appel **empile** une case avec **ses propres** variables (le `n` de `somme(3)` n'est pas le même que celui de `somme(2)`).
- Quand un appel se termine, sa case est **dépilée**, et on reprend l'appel du dessous là où il s'était arrêté.
- La pile a une taille limitée : environ **1000** appels en Python. C'est pour ça que `somme(5000)` provoque une `RecursionError`, alors que la même somme avec une boucle marche très bien.

---
# 5. Méthode : écrire une fonction récursive en 3 questions

Exemple : la **factorielle** $n! = 1 \times 2 \times \cdots \times n$ (et par convention $0! = 1$).

1. **Quel est le cas le plus simple, dont je connais directement la réponse ?** → `n == 0`, la réponse est `1`.
2. **Si je connais la réponse pour un problème un peu plus petit, comment j'obtiens la réponse pour le mien ?** → si je connais $(n-1)!$, alors $n! = n \times (n-1)!$.
3. **Est-ce que chaque appel se rapproche du cas de base ?** → oui, `n` diminue de 1 à chaque appel et finit par valoir `0` (si `n` est positif au départ).

Il ne reste qu'à traduire :

In [4]:
def factorielle(n):
    if n == 0:                           # question 1
        return 1
    return n * factorielle(n - 1)        # question 2


print(factorielle(5))
print(factorielle(20))

120
2432902008176640000


#### 💡 À retenir
- La question 2 est la plus importante : il faut **faire confiance** à l'appel récursif. On ne cherche pas à dérouler tous les appels dans sa tête : on **suppose** que `factorielle(n - 1)` renvoie la bonne réponse, et on se demande seulement comment s'en servir.
- La question 3 garantit que la fonction **se termine**. Au bac, on demande souvent de **justifier** qu'une fonction récursive termine : on explique que le paramètre diminue à chaque appel et finit par atteindre le cas de base.

---
# 6. Récursif ou boucle ? Comment savoir

Tout ce qu'on fait avec une boucle peut se faire avec la récursivité, et inversement. Voici la même somme écrite avec une boucle :

In [5]:
def somme_boucle(n):
    total = 0
    for k in range(1, n + 1):
        total = total + k
    return total


print(somme_boucle(4), somme(4))       # même résultat

10 10


Alors, quand faut-il vraiment la récursivité ?

**Par défaut, on écrit une boucle.** On passe en récursif si on repère **un de ces signes** :

> **Signe 1 — Le problème contient le même problème, en plus petit.**
> Pour résoudre mon problème, je dois résoudre **le même** problème sur un morceau.
>
> **Signe 2 — Je coupe en morceaux que je traite tous pareil.**
> Je coupe en 2 (ou plus), je traite chaque morceau **de la même façon**, puis je recolle.
> *Exemples : le tri fusion (deux moitiés), les tours de Hanoï, le « diviser pour régner ».*
>
> **Signe 3 — Je dois explorer plusieurs directions ou plusieurs choix.**
> À chaque étape, il y a plusieurs possibilités, et il faut **toutes** les essayer.
> *Exemples : les 4 voisins d'une case (pot de peinture, labyrinthe), tous les chemins dans une grille.*
>
> **Signe 4 — Des choses emboîtées les unes dans les autres.**
> Comme des poupées russes : des dossiers dans des dossiers, des parenthèses dans des parenthèses, des arbres (en Terminale).

**Le test pratique :** est-ce que j'arrive à l'écrire avec des boucles ? Si je bloque parce qu'il me faudrait **des boucles dans des boucles, sans savoir combien** à l'avance, c'est qu'il faut la récursivité. *(Pour Hanoï avec 3 disques, il faudrait 3 niveaux de boucles ; avec 10 disques, 10 niveaux : le nombre dépend de `n`, aucune boucle écrite à l'avance ne convient.)*

**Deux pièges, où on revient à une boucle :**
- le **même calcul est refait** plein de fois (exemple : la suite de Fibonacci) ;
- il y aurait **plus de 1000 appels empilés** (Python s'arrête avec `RecursionError`).

Et bien sûr : si l'énoncé dit « écrire une fonction **récursive** », pas le choix !

**À retenir en une phrase :** par défaut, une boucle. Si le problème contient le même problème en plus petit, ou plusieurs directions à explorer, je passe en récursif.

⚠️ Dans la feuille d'exercices, les parties A et C font utiliser la récursivité sur des problèmes simples, où une boucle irait aussi bien : ce sont des **entraînements** pour apprendre le mécanisme (et certains sujets du bac le demandent). La partie B montre les problèmes où la récursivité est **vraiment** nécessaire.

---
# 7. Récursivité sur une liste ou une chaîne : avec un indice

Pour travailler sur une liste de façon récursive, on ajoute un paramètre `i` : la fonction traite **la partie de la liste qui commence à l'indice `i`**.

Exemple : la somme des éléments de `t` à partir de l'indice `i`.
- **Cas de base :** si `i == len(t)`, il n'y a plus aucun élément : la somme vaut `0`.
- **Cas récursif :** la somme à partir de `i`, c'est `t[i]` plus la somme à partir de `i + 1`.

In [6]:
def somme_liste(t, i):
    if i == len(t):
        return 0
    return t[i] + somme_liste(t, i + 1)


notes = [12, 15, 9, 14]
print(somme_liste(notes, 0))           # on commence à l'indice 0 : toute la liste

50


#### 💡 À retenir
- Le premier appel se fait avec `i = 0` : on traite toute la liste.
- À chaque appel, `i` augmente de 1 : la partie restante est plus courte d'un élément, jusqu'à être vide (`i == len(t)`).
- La liste `t` elle-même n'est **jamais modifiée ni recopiée** : seul l'indice change.
- Ça marche de la même façon pour une chaîne de caractères : `ch[i]` est le caractère d'indice `i`.

---
# Récapitulatif

| Notion | À retenir |
|:---|:---|
| Fonction récursive | une fonction qui s'appelle elle-même sur un problème **plus petit** |
| Cas de base | situation simple où on répond **sans** appel récursif. **Obligatoire.** |
| Cas récursif | on se ramène à un problème plus petit, qui se rapproche du cas de base |
| Terminaison | justifier que le paramètre se rapproche du cas de base à chaque appel |
| Pile d'appels | les appels en attente s'empilent ; limite d'environ 1000 appels (`RecursionError`) |
| Méthode | 1) cas le plus simple ? 2) comment utiliser la réponse du problème plus petit ? 3) se rapproche-t-on du cas de base ? |
| Sur une liste ou une chaîne | ajouter un paramètre indice `i` ; cas de base `i == len(t)` |
| Récursif ou boucle ? | par défaut une boucle ; récursif si le problème contient le même problème en plus petit, se coupe en morceaux traités pareil, ou demande d'explorer plusieurs directions ; pièges : même calcul refait, plus de 1000 appels |

**Et maintenant :** la feuille d'exercices `02_Recursivite_Exercices.ipynb`, du plus facile ⭐ au plus difficile ⭐⭐⭐⭐. Les exercices marqués 📚 sont tirés de sujets du **bac** (épreuve écrite ou épreuve pratique).